In [15]:
#!pip install plotnine

In [32]:
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np
from plotnine import ggplot, aes, geom_point, facet_grid, labs, scale_y_continuous, labeller, as_labeller, scale_x_continuous
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [20]:
# read in data
data = pd.read_csv("../data/rl_policies/tqc_clean.csv")

In [21]:
# get rid of t = 0:
data = data[data["t"] > 0]

In [22]:
# remove anomolous biomass data
data = data[(data['biomass'] != -1) & (data['biomass'] <= -0.46)]

In [23]:
# subset to relevant columns
subset = data[['months','act0', 'act1', 'CPUE', 'biomass']]



In [24]:
# set K range and loop through months and actions to find best k and cluster data
K_range = range(2, 7)
months = subset['months'].unique()
actions = ['act0', 'act1']
all_centroids = []
all_labeled = []

for month in months:
    data_month = subset[subset['months'] == month].drop(columns=['months'])
    
    for action in actions:
        other_action = 'act1' if action == 'act0' else 'act0'
        X = data_month.drop(columns=[other_action])
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        scores = []
        for k in K_range:
            km = KMeans(n_clusters=k, n_init=30, random_state=42).fit(X_scaled)
            scores.append(silhouette_score(X_scaled, km.labels_))
        
        best_k = K_range[np.argmax(scores)]
        kmeans = KMeans(n_clusters=best_k, n_init=30, random_state=42).fit(X_scaled)
        
        # build labeled chunk
        chunk = X.copy().reset_index(drop=True)
        chunk['month'] = month
        chunk['action'] = action
        chunk['cluster'] = kmeans.labels_
        all_labeled.append(chunk)
        
        # store centroids
        centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
        centroids_df = pd.DataFrame(centroids_original, columns=X.columns)
        centroids_df['month'] = month
        centroids_df['action'] = action
        centroids_df['cluster'] = range(best_k)
        all_centroids.append(centroids_df)

labeled = pd.concat(all_labeled, ignore_index=True)
centroids = pd.concat(all_centroids, ignore_index=True)



In [29]:
labeled.head()

,act0,CPUE,biomass,month,action,cluster,act1
0,-0.964028,-0.994372,-0.781113,5,act0,1,NaN
1,-0.964028,-0.980309,-0.659227,5,act0,0,NaN
2,-0.964028,-0.982911,-0.630332,5,act0,0,NaN
3,-0.964028,-0.980258,-0.644311,5,act0,0,NaN
4,-0.964028,-0.985185,-0.622753,5,act0,0,NaN


In [25]:
# save labeled data and centroids
labeled.to_csv("labeled.csv", index=False)
centroids.to_csv("centroids.csv", index=False)


In [26]:
def find_closest_actions(CPUE, biomass, month, centroids):
    results = {}
    for action in ['act0', 'act1']:
        subset = centroids[(centroids['month'] == month) & (centroids['action'] == action)].copy()
        subset['dist'] = ((subset['CPUE'] - CPUE)**2 + (subset['biomass'] - biomass)**2)**0.5
        closest = subset.loc[subset['dist'].idxmin()]
        results[action] = closest[action]
    return results['act0'], results['act1']


class CentroidAgent:
    """Agent that selects actions by finding the nearest centroid for the current observation.

    Expects observations of the form {"crabs": np.array([CPUE, biomass]), "months": month},
    i.e. observation_type='count-biomass-time'.
    """
    def __init__(self, centroids: pd.DataFrame, env):
        self.centroids = centroids
        self.env = env

    def predict(self, observation, **kwargs):
        CPUE = float(observation["crabs"][0])
        biomass = float(observation["crabs"][1])
        month = int(observation["months"])
        act0, act1 = find_closest_actions(CPUE, biomass, month, self.centroids)
        return np.array([act0, act1], dtype=np.float32), {}

# example usage:
# centroid_agent = CentroidAgent(centroids=centroids, env=evalEnv)
# centroid_plot_agent = plot_agent(env_sim_df=None,
#                                  agent_name='centroid_agent',
#                                  env=evalEnv,
#                                  agent=centroid_agent,
#                                  save_dir='.')

In [37]:
month_names = {"4": "Apr", "5": "May",
               "6": "June", "7": "July", "8": "Aug",
               "9": "Sep", "10": "Oct"}
action_names = {"act0": "Minnow traps", "act1": "Fukui traps"}

month_order = ["Apr", "May", "June", "July", "Aug", "Sep", "Oct"]

plot_data = labeled.copy()
plot_data['month'] = pd.Categorical(
    plot_data['month'].astype(str).map(month_names),
    categories=month_order,
    ordered=True
)
plot_data['action'] = plot_data['action'].map(action_names)


p = (
    ggplot(plot_data, aes(x='biomass', y='CPUE', color='factor(cluster)'))
    + geom_point(alpha=0.4, size=0.5)
    + facet_grid('action ~ month')
    + labs(color='Cluster')
)
p.save('../figures/clustering.png', dpi=300, height = 3, width = 8)


/home/jovyan/miniconda/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/jovyan/miniconda/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: ../figures/clustering.png


In [30]:
centroids

,act0,CPUE,biomass,month,action,cluster,act1
0,-0.964028,-0.985510,-0.665928,5,act0,0,NaN
1,-0.964028,-0.996856,-0.783867,5,act0,1,NaN
2,NaN,-0.985525,-0.665823,5,act1,0,-0.039312
3,NaN,-0.996605,-0.784883,5,act1,1,-0.758695
4,-0.964028,-0.988591,-0.612539,6,act0,0,NaN
5,-0.964028,-0.996876,-0.729839,6,act0,1,NaN
6,NaN,-0.988591,-0.612539,6,act1,0,0.303038
7,NaN,-0.996876,-0.729839,6,act1,1,0.119512
8,-0.964028,-0.991417,-0.537867,7,act0,0,NaN
9,-0.964028,-0.997221,-0.664579,7,act0,1,NaN


In [10]:
max(centroids['act0'])

0.7765293592878563

In [11]:
min(centroids['act0'])

-0.9640276